# 04_guest_role_vs_rejected_anonymous — "no token" is not one single outcome

The resolver always calls `auth_coordinator.process(request)` — but that is not the same as "requires a real logged-in user". Two genuinely different things can happen to an unauthenticated request, and this notebook shows both side by side so they are never confused:

- **(a)** A coordinator that resolves missing credentials to a real, legitimate anonymous `Context` (the same thing `NoAuthCoordinator` always does) lets the request reach the machine normally. A `@check_roles(GuestRole)` action then gets an honest `kind: "success"` result — `GuestRole` is evaluated exactly like any other role, not special-cased inside the resolver.
- **(b)** A coordinator whose `process()` genuinely returns `None` (e.g. credentials were supplied but are invalid) never reaches the machine at all — the resolver answers `403` straight away.

This is the actual `FastApiAdapter.build()` + `TestClient`, not just `machine.check_access_decide()` directly — the "`None` vs. resolved `Context`" distinction lives in the HTTP-layer resolver, not in the machine.

**What's new**

| Concept | Description |
|---|---|
| `GuestRole` | A real role — evaluated like any other, not a resolver special case |
| `process()` returning `None` | The one case the resolver rejects, before ever reaching the machine (`403`) |

In [ ]:
!pip install aoa-action-machine aoa-fastapi-adapter

In [ ]:
from unittest.mock import AsyncMock

from fastapi.testclient import TestClient
from pydantic import Field

from aoa.action_machine.auth.guest_role import GuestRole
from aoa.action_machine.context.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine
from aoa.fastapi.adapter import FastApiAdapter

## A public, guest-accessible action

In [ ]:
class StoreDomain(BaseDomain):
    name = "store"
    description = "Store domain"


class PublicParams(BaseParams):
    pass


class PublicResult(BaseResult):
    message: str = Field(description="Response message")


@meta(description="Browse the public catalog", domain=StoreDomain)
@check_roles(GuestRole)  # open to anyone — declared, not a default
class BrowseCatalogAction(BaseAction[PublicParams, PublicResult]):

    @summary_aspect("Return the catalog")
    async def browse_summary(self, params, state, box, connections):
        return PublicResult(message="catalog")

## A helper that lets us control what `auth_coordinator.process()` returns

In [ ]:
def make_client(*, resolved_context):
    machine = ActionProductMachine()
    auth = AsyncMock()
    auth.process.return_value = resolved_context  # simulates one coordinator's decision
    adapter = FastApiAdapter(machine=machine, auth_coordinator=auth)
    adapter.get("/actions/browse-catalog", BrowseCatalogAction)
    return TestClient(adapter.build())

## (a) Resolved anonymous Context — a real, honest verdict

In [ ]:
resolve_body = {"version": 1, "items": [{"operation": "GET /actions/browse-catalog", "params": {}}]}

guest_client = make_client(resolved_context=Context())
guest_response = guest_client.post("/permissions/resolve", json=resolve_body)
result = guest_response.json()["results"][0]
print(f"status = {guest_response.status_code}")
print(f"result = {result}")  # {'kind': 'success', 'reason': ''}

## (b) process() returns None — rejected before the machine ever sees it

In [ ]:
rejected_client = make_client(resolved_context=None)
rejected_response = rejected_client.post("/permissions/resolve", json=resolve_body)
print(f"status = {rejected_response.status_code}")  # 403 — AuthorizationError's handler